In [233]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.distributions.empirical_distribution import ECDF

# Ahora importa el módulo
from RRAM.Representate import config_ax, setup_paper_plt, config_ax_IV
    
# Configuración de la figura
setup_paper_plt(plt, latex=True, scaling=2.2)

#### CDF Set

In [234]:
ruta_datos = Path.cwd() / "Datos_Experimentales" / "Medidas_Experimentales_RRAM"
ruta_figuras = Path.cwd() / "Datos_Experimentales" / "V_Set"

In [235]:
results_path = ruta_figuras / "V_set_simulacion_2_CF.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_creacion_1", "f8"),  # Float de 64 bits
        ("V_creacion_2", "f8"),  # Float de 64 bits
        # ("V_creacion_3", "f8"),  # Float de 64 bits
        # ("V_creacion_4", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=[
        "Archivo",
        "V_creacion_1",
        "V_creacion_2",
        # "V_creacion_3",
        # "V_creacion_4",
    ],  # NO se refiere a que se ha creado el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim = pd.DataFrame(resultados_txt)

df_resultados_sim["Numero"] = (
    df_resultados_sim["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

print(df_resultados_sim)

                    Archivo  V_creacion_1  V_creacion_2  Numero
0      log_simulacion_1.log        0.4082        0.5510       1
1     log_simulacion_10.log        0.4341        0.5719      10
2    log_simulacion_100.log        0.4644        0.5474     100
3    log_simulacion_101.log        0.4129        0.5083     101
4    log_simulacion_102.log        0.4266        0.5615     102
..                      ...           ...           ...     ...
195   log_simulacion_95.log        0.4417        0.6521      95
196   log_simulacion_96.log        0.3543        0.5485      96
197   log_simulacion_97.log        0.3587        0.5681      97
198   log_simulacion_98.log        0.4214        0.6087      98
199   log_simulacion_99.log        0.4206        0.5418      99

[200 rows x 4 columns]


In [236]:
results_path = ruta_figuras / "V_set_experimental.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_set_derivada", "f8"),  # Float de 64 bits
        ("V_set_elbow", "f8"),
    ],
    encoding="utf-8",
    names=["Archivo", "V_set_derivada_V", "V_set_elbow_V"],
)

# Crear un DataFrame para una mejor visualización
df_resultados_exp = pd.DataFrame(resultados_txt)

df_resultados_exp["Numero"] = (
    df_resultados_exp["Archivo"].str.extract(r"Cycle_p_(\d+)", expand=False).astype(int)
)

print(df_resultados_exp)

          Archivo  V_set_derivada_V  V_set_elbow_V  Numero
0    Cycle_p_1000              0.53           0.52    1000
1    Cycle_p_1001              0.50           0.54    1001
2    Cycle_p_1002              0.49           0.54    1002
3    Cycle_p_1003              0.47           0.47    1003
4    Cycle_p_1004              0.51           0.50    1004
..            ...               ...            ...     ...
895   Cycle_p_995              0.52           0.51     995
896   Cycle_p_996              0.55           0.53     996
897   Cycle_p_997              0.51           0.50     997
898   Cycle_p_998              0.52           0.52     998
899   Cycle_p_999              0.54           0.54     999

[900 rows x 4 columns]


In [237]:
V_set_sim_CF_1 = df_resultados_sim["V_creacion_1"]
V_set_sim_CF_2 = df_resultados_sim["V_creacion_2"]
# V_set_sim_CF_all = df_resultados_sim["V_creacion_4"]

V_set_exp_der = df_resultados_exp["V_set_derivada_V"]
V_set_max_int = df_resultados_exp["V_set_elbow_V"]

# ECDF objects
ecdf_sim_CF_1 = ECDF(V_set_sim_CF_1)
ecdf_sim_CF_2 = ECDF(V_set_sim_CF_2)
# ecdf_sim_CF_all = ECDF(V_set_sim_CF_all)

ecdf_exp_der = ECDF(V_set_exp_der)
ecdf_exp_max_int = ECDF(V_set_max_int)

fig, ax = plt.subplots(figsize=(12, 9))
config_ax_IV(ax)

# Common range
min_val = 0
max_val = 1.4
x = np.linspace(min_val, max_val, 1000)

# Evaluate ECDFs
y_sim_1 = ecdf_sim_CF_1(x)
y_sim_2 = ecdf_sim_CF_2(x)
# y_sim_all = ecdf_sim_CF_all(x)

y_der = ecdf_exp_der(x)
y_max_int = ecdf_exp_max_int(x)

# Plot ECDFs
ax.step(x, y_sim_1, where="post", label="Sim. - 1 CF", linewidth=2.5)
ax.step(x, y_sim_2, where="post", label="Sim. - 2 CF", linewidth=2.5)
# ax.step(x, y_sim_all, where="post", label="Sim. - All CF", linewidth=2.5)

ax.scatter(
    x,
    y_der,
    label="Exp. - Derivative",
    linewidth=1,
    s=50,
    marker="o",
    # color="green",
)
ax.scatter(
    x,
    y_max_int,
    label="Exp. - Elbow",
    linewidth=1,
    s=50,
    marker="s",
    # color="red",
)

# Labels in English
ax.set_xlabel(r"Set voltage (\si{\volt})", fontsize=40)
ax.set_ylabel("CDF", fontsize=40)
# ax.grid(True)
plt.legend(loc="best", fontsize=26)
fig.savefig(str(Path.cwd()) + "/CDF_set_2_CF.pdf", dpi=300, bbox_inches="tight")
plt.close(fig)

#### CDF Reset

Tanto el set como el reset deben mantener la escala es decir, ambos plots deben comenzar en 0 y acabar en 1.4 V


In [238]:
ruta_datos = Path.cwd() / "Datos_Experimentales" / "Medidas_Experimentales_RRAM"
ruta_figuras = Path.cwd() / "Datos_Experimentales" / "V_Reset"

In [239]:
# Obtener V_set de los archivos de simulación
results_path = ruta_figuras / "V_reset_simulacion_2_CF.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_rotura_1", "f8"),  # Float de 64 bits
        ("V_rotura_2", "f8"),  # Float de 64 bits
        # ("V_rotura_3", "f8"),  # Float de 64 bits
        # ("V_rotura_4", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=[
        "Archivo",
        "V_rotura_1",
        "V_rotura_2",
        # "V_rotura_3",
        # "V_rotura_4",
    ],  # NO se refiere a que se ha roto el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim = pd.DataFrame(resultados_txt)
df_resultados_sim["Numero"] = (
    df_resultados_sim["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

# Elimino la primera columna del dataframe
df_resultados_sim = df_resultados_sim.drop(columns=["Archivo"])

# Muevo la primera columna al de numero al inicio
cols = df_resultados_sim.columns.tolist()
cols = [cols[-1]] + cols[:-1]
df_resultados_sim = df_resultados_sim[cols]
print(df_resultados_sim)


     Numero  V_rotura_1  V_rotura_2
0         1     -1.1435     -1.1817
1        10     -1.1204     -1.1248
2       100     -1.1425     -1.1900
3       101     -1.1659     -1.1677
4       102     -1.1518     -1.1696
..      ...         ...         ...
195      95     -1.0000     -1.2792
196      96     -1.1103     -1.1938
197      97     -1.1556     -1.1711
198      98     -1.1179     -1.1672
199      99     -1.1497     -1.2347

[200 rows x 3 columns]


In [240]:
results_path = ruta_figuras / "V_Reset_experimental.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_reset_derivada", "f8"),  # Float de 64 bits
        ("V_reset_max_intensidad", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=["Archivo", "V_reset_derivada_V", "V_reset_max_intensidad_V"],
)

# Crear un DataFrame para una mejor visualización
df_resultados = pd.DataFrame(resultados_txt)

df_resultados["Numero"] = (
    df_resultados["Archivo"].str.extract(r"Cycle_n_(\d+)", expand=False).astype(int)
)
print(df_resultados)


           Archivo  V_reset_derivada_V  V_reset_max_intensidad_V  Numero
0     Cycle_n_1000               -1.14                     -1.07    1000
1     Cycle_n_1001               -1.13                     -1.07    1001
2     Cycle_n_1002               -1.12                     -1.04    1002
3     Cycle_n_1003               -1.14                     -1.07    1003
4     Cycle_n_1004               -1.12                     -1.05    1004
...            ...                 ...                       ...     ...
3296   Cycle_n_995               -1.13                     -1.06     995
3297   Cycle_n_996               -1.15                     -1.07     996
3298   Cycle_n_997               -1.15                     -1.09     997
3299   Cycle_n_998               -1.14                     -1.08     998
3300   Cycle_n_999               -1.14                     -1.07     999

[3301 rows x 4 columns]


In [241]:
V_reset_sim_CF_1 = abs(df_resultados_sim["V_rotura_1"])
V_reset_sim_CF_2 = abs(df_resultados_sim["V_rotura_2"])
# V_reset_sim_CF_all = abs(df_resultados_sim["V_rotura_4"])

V_reset_exp_der = abs(df_resultados["V_reset_derivada_V"])
V_reset_max_int = abs(df_resultados["V_reset_max_intensidad_V"])

# ECDF objects
ecdf_sim_CF_1 = ECDF(V_reset_sim_CF_1)
ecdf_sim_CF_2 = ECDF(V_reset_sim_CF_2)
# ecdf_sim_CF_all = ECDF(V_reset_sim_CF_all)

ecdf_exp_der = ECDF(V_reset_exp_der)
ecdf_exp_max_int = ECDF(V_reset_max_int)

fig, ax = plt.subplots(figsize=(12, 9))
config_ax_IV(ax)

# Common range
min_val = 0
max_val = 1.4

x = np.linspace(min_val, max_val, 1000)

# Evaluate ECDFs
y_sim_1 = ecdf_sim_CF_1(x)
y_sim_2 = ecdf_sim_CF_2(x)
# y_sim_all = ecdf_sim_CF_all(x)
y_der = ecdf_exp_der(x)
y_max_int = ecdf_exp_max_int(x)

# Plot ECDFs
ax.step(x, y_sim_1, where="post", label="Sim. - 1 CF", linewidth = 2.5)
ax.step(x, y_sim_2, where="post", label="Sim. - 2 CF", linewidth = 2.5)
# ax.step(x, y_sim_all, where="post", label="Sim. - All CF", linewidth = 2.5)
ax.scatter(x, y_der, s = 50, marker = "o", label="Exp. - Derivative", linewidth = 1)
ax.scatter(x, y_max_int, s = 50, marker = "s", label="Exp. - Max Current", linewidth = 1)

# Labels in English
ax.set_xlabel(r"|Reset voltage| (\si{\volt})", fontsize=40)
ax.set_ylabel("CDF", fontsize=40)
# ax.grid(True)
plt.legend(loc="upper left", fontsize=20)

fig.savefig(str(Path.cwd()) + "/CDF_reset_2_CF.pdf", dpi=300, bbox_inches="tight")
plt.close(fig)

#### CDF para 2 y 4 Filamentos SET

In [242]:
ruta_set = Path.cwd() / "Datos_Experimentales" / "V_Set"

results_4_CF_path = ruta_set / "V_set_simulacion_4_CF.txt"

resultados_txt_4_CF = np.genfromtxt(
    results_4_CF_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_creacion_1", "f8"),  # Float de 64 bits
        ("V_creacion_2", "f8"),  # Float de 64 bits
        ("V_creacion_3", "f8"),  # Float de 64 bits
        ("V_creacion_4", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=[
        "Archivo",
        "V_creacion_1",
        "V_creacion_2",
        "V_creacion_3",
        "V_creacion_4",
    ],  # NO se refiere a que se ha creado el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim_4_CF = pd.DataFrame(resultados_txt_4_CF)


df_resultados_sim_4_CF["Numero"] = (
    df_resultados_sim_4_CF["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

print(df_resultados_sim_4_CF)

                    Archivo  V_creacion_1  V_creacion_2  V_creacion_3  \
0      log_simulacion_1.log        0.5610        0.5817        0.6301   
1     log_simulacion_10.log        0.5223        0.6181        0.6419   
2    log_simulacion_100.log        0.5452        0.5667        0.5708   
3    log_simulacion_101.log        0.5138        0.5730        0.6747   
4    log_simulacion_102.log        0.5522        0.6478        0.6752   
..                      ...           ...           ...           ...   
579   log_simulacion_95.log        0.5477        0.5872        0.6194   
580   log_simulacion_96.log        0.4869        0.5737        0.6132   
581   log_simulacion_97.log        0.5321        0.5489        0.5873   
582   log_simulacion_98.log        0.5229        0.5425        0.6541   
583   log_simulacion_99.log        0.5060        0.5755        0.6062   

     V_creacion_4  Numero  
0          0.8183       1  
1          0.6829      10  
2          0.6948     100  
3          

In [243]:
results_2_CF_path = ruta_set / "V_set_simulacion_2_CF.txt"

resultados_txt_2_CF = np.genfromtxt(
    results_2_CF_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_creacion_1", "f8"),  # Float de 64 bits
        ("V_creacion_2", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=[
        "Archivo",
        "V_creacion_1",
        "V_creacion_2",
    ],  # NO se refiere a que se ha creado el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim_2_CF = pd.DataFrame(resultados_txt_2_CF)


df_resultados_sim_2_CF["Numero"] = (
    df_resultados_sim_2_CF["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

print(df_resultados_sim_2_CF)

                    Archivo  V_creacion_1  V_creacion_2  Numero
0      log_simulacion_1.log        0.4082        0.5510       1
1     log_simulacion_10.log        0.4341        0.5719      10
2    log_simulacion_100.log        0.4644        0.5474     100
3    log_simulacion_101.log        0.4129        0.5083     101
4    log_simulacion_102.log        0.4266        0.5615     102
..                      ...           ...           ...     ...
195   log_simulacion_95.log        0.4417        0.6521      95
196   log_simulacion_96.log        0.3543        0.5485      96
197   log_simulacion_97.log        0.3587        0.5681      97
198   log_simulacion_98.log        0.4214        0.6087      98
199   log_simulacion_99.log        0.4206        0.5418      99

[200 rows x 4 columns]


In [244]:
results_path = ruta_set / "V_set_experimental.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_set_derivada", "f8"),  # Float de 64 bits
        ("V_set_elbow", "f8"),
    ],
    encoding="utf-8",
    names=["Archivo", "V_set_derivada_V", "V_set_elbow_V"],
)

# Crear un DataFrame para una mejor visualización
df_resultados_exp = pd.DataFrame(resultados_txt)

df_resultados_exp["Numero"] = (
    df_resultados_exp["Archivo"].str.extract(r"Cycle_p_(\d+)", expand=False).astype(int)
)

print(df_resultados_exp)

          Archivo  V_set_derivada_V  V_set_elbow_V  Numero
0    Cycle_p_1000              0.53           0.52    1000
1    Cycle_p_1001              0.50           0.54    1001
2    Cycle_p_1002              0.49           0.54    1002
3    Cycle_p_1003              0.47           0.47    1003
4    Cycle_p_1004              0.51           0.50    1004
..            ...               ...            ...     ...
895   Cycle_p_995              0.52           0.51     995
896   Cycle_p_996              0.55           0.53     996
897   Cycle_p_997              0.51           0.50     997
898   Cycle_p_998              0.52           0.52     998
899   Cycle_p_999              0.54           0.54     999

[900 rows x 4 columns]


In [245]:
V_set_sim_2_CF_1 = df_resultados_sim_2_CF["V_creacion_1"]
V_set_sim_2_CF_2 = df_resultados_sim_2_CF["V_creacion_2"]

V_set_sim_4_CF_1 = df_resultados_sim_4_CF["V_creacion_1"]
V_set_sim_4_CF_2 = df_resultados_sim_4_CF["V_creacion_2"]
V_set_sim_4_CF_all = df_resultados_sim_4_CF["V_creacion_4"]

V_set_exp_der = df_resultados_exp["V_set_derivada_V"]
V_set_elbow = df_resultados_exp["V_set_elbow_V"]

# ECDF objects
ecdf_sim_2_CF_1 = ECDF(V_set_sim_2_CF_1)
ecdf_sim_2_CF_2 = ECDF(V_set_sim_2_CF_2)

ecdf_sim_4_CF_1 = ECDF(V_set_sim_4_CF_1)
ecdf_sim_4_CF_2 = ECDF(V_set_sim_4_CF_2)
ecdf_sim_4_CF_all = ECDF(V_set_sim_4_CF_all)

ecdf_exp_der = ECDF(V_set_exp_der)
ecdf_exp_elbow = ECDF(V_set_elbow)

fig, ax = plt.subplots(figsize=(12, 9))
config_ax_IV(ax)

# Common range
min_val = 0
max_val = 1.4
x = np.linspace(min_val, max_val, 1000)

# Evaluate 2 CF ECDFs
y_sim_2_CF_1 = ecdf_sim_2_CF_1(x)
y_sim_2_CF_2 = ecdf_sim_2_CF_2(x)

# Evaluate 4 CF ECDFs
y_sim_4_CF_1 = ecdf_sim_4_CF_1(x)
y_sim_4_CF_2 = ecdf_sim_4_CF_2(x)
y_sim_4_CF_all = ecdf_sim_4_CF_all(x)

# Evaluate experimental ECDFs
y_der = ecdf_exp_der(x)
y_elbow = ecdf_exp_elbow(x)

# Plot ECDFs
ax.step(
    x,
    y_sim_4_CF_1,
    where="post",
    label="Sim. - 1 CF (4 CF)",
    linewidth=2.5,
    linestyle="dashed",
)
ax.step(
    x,
    y_sim_4_CF_2,
    where="post",
    label="Sim. - 2 CF (4 CF)",
    linewidth=2.5,
    linestyle="dashed",
)
ax.step(
    x,
    y_sim_4_CF_all,
    where="post",
    label="Sim. - All CF (4 CF)",
    linewidth=2.5,
    linestyle="dashed",
)

ax.step(x, y_sim_2_CF_1, where="post", label="Sim. - 1 CF (2 CF)", linewidth=2.5)
ax.step(x, y_sim_2_CF_2, where="post", label="Sim. - 2 CF (2 CF)", linewidth=2.5)

ax.scatter(
    x,
    y_der,
    label="Exp. - Derivative",
    linewidth=1,
    s=50,
    marker="o",
)
ax.scatter(x, y_elbow,label="Exp. - Elbow", linewidth=1, s=50, marker="s")

# Labels in English
ax.set_xlabel(r"Set voltage (\si{\volt})", fontsize=32)
ax.set_ylabel("CDF", fontsize=32)
plt.legend(loc="best", fontsize=23)
fig.savefig(str(Path.cwd()) + "/CDF_set_ambos.pdf", dpi=300, bbox_inches="tight")
plt.close(fig)

#### CDF para 2 y 4 Filementos RESET

In [246]:
ruta_reset = Path.cwd() / "Datos_Experimentales" / "V_Reset"
results_4_CF_path = ruta_reset / "V_reset_simulacion_4_CF.txt"

resultados_txt_4_CF = np.genfromtxt(
    results_4_CF_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_rotura_1", "f8"),  # Float de 64 bits
        ("V_rotura_2", "f8"),  # Float de 64 bits
        ("V_rotura_3", "f8"),  # Float de 64 bits
        ("V_rotura_4", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=[
        "Archivo",
        "V_rotura_1",
        "V_rotura_2",
        "V_rotura_3",
        "V_rotura_4",
    ],  # NO se refiere a que se ha roto el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim_4_CF = pd.DataFrame(resultados_txt_4_CF)


df_resultados_sim_4_CF["Numero"] = (
    df_resultados_sim_4_CF["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

print(df_resultados_sim_4_CF)

                    Archivo  V_rotura_1  V_rotura_2  V_rotura_3  V_rotura_4  \
0      log_simulacion_1.log     -1.1430     -1.1826     -1.1948     -1.3439   
1     log_simulacion_10.log     -1.1302     -1.1642     -1.2576     -1.2802   
2    log_simulacion_102.log     -1.1094     -1.1379     -1.1894     -1.2776   
3    log_simulacion_103.log     -1.1277     -1.2057     -1.2351     -1.2687   
4    log_simulacion_104.log     -1.1535     -1.2048     -1.2158     -1.2260   
..                      ...         ...         ...         ...         ...   
162   log_simulacion_95.log     -1.1367     -1.1381     -1.2571     -1.3651   
163   log_simulacion_96.log     -1.1490     -1.1574     -1.2396     -1.3149   
164   log_simulacion_97.log     -1.1257     -1.1795     -1.1901     -1.3294   
165   log_simulacion_98.log     -1.1208     -1.1522     -1.1883     -1.2902   
166   log_simulacion_99.log     -1.1423     -1.1593     -1.2457     -1.3450   

     Numero  
0         1  
1        10  
2       1

In [247]:
results_2_CF_path = ruta_reset / "V_reset_simulacion_2_CF.txt"

resultados_txt_2_CF = np.genfromtxt(
    results_2_CF_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_rotura_1", "f8"),  # Float de 64 bits
        ("V_rotura_2", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=[
        "Archivo",
        "V_rotura_1",
        "V_rotura_2",
    ],  # NO se refiere a que se ha creado el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim_2_CF = pd.DataFrame(resultados_txt_2_CF)

df_resultados_sim_2_CF["Numero"] = (
    df_resultados_sim_2_CF["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

print(df_resultados_sim_2_CF)

                    Archivo  V_rotura_1  V_rotura_2  Numero
0      log_simulacion_1.log     -1.1435     -1.1817       1
1     log_simulacion_10.log     -1.1204     -1.1248      10
2    log_simulacion_100.log     -1.1425     -1.1900     100
3    log_simulacion_101.log     -1.1659     -1.1677     101
4    log_simulacion_102.log     -1.1518     -1.1696     102
..                      ...         ...         ...     ...
195   log_simulacion_95.log     -1.0000     -1.2792      95
196   log_simulacion_96.log     -1.1103     -1.1938      96
197   log_simulacion_97.log     -1.1556     -1.1711      97
198   log_simulacion_98.log     -1.1179     -1.1672      98
199   log_simulacion_99.log     -1.1497     -1.2347      99

[200 rows x 4 columns]


In [248]:
results_path = ruta_reset / "V_reset_experimental.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_reset_derivada", "f8"),  # Float de 64 bits
        ("V_reset_max_intensidad", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=["Archivo", "V_reset_derivada_V", "V_reset_max_intensidad_V"],
)

# Crear un DataFrame para una mejor visualización
df_resultados_exp = pd.DataFrame(resultados_txt)

df_resultados_exp["Numero"] = (
    df_resultados_exp["Archivo"].str.extract(r"Cycle_n_(\d+)", expand=False).astype(int)
)

print(df_resultados_exp)

           Archivo  V_reset_derivada_V  V_reset_max_intensidad_V  Numero
0     Cycle_n_1000               -1.14                     -1.07    1000
1     Cycle_n_1001               -1.13                     -1.07    1001
2     Cycle_n_1002               -1.12                     -1.04    1002
3     Cycle_n_1003               -1.14                     -1.07    1003
4     Cycle_n_1004               -1.12                     -1.05    1004
...            ...                 ...                       ...     ...
3296   Cycle_n_995               -1.13                     -1.06     995
3297   Cycle_n_996               -1.15                     -1.07     996
3298   Cycle_n_997               -1.15                     -1.09     997
3299   Cycle_n_998               -1.14                     -1.08     998
3300   Cycle_n_999               -1.14                     -1.07     999

[3301 rows x 4 columns]


In [251]:
V_reset_sim_2_CF_1 = abs(df_resultados_sim_2_CF["V_rotura_1"])
V_reset_sim_2_CF_2 = abs(df_resultados_sim_2_CF["V_rotura_2"])

V_reset_sim_4_CF_1 = abs(df_resultados_sim_4_CF["V_rotura_1"])
V_reset_sim_4_CF_2 = abs(df_resultados_sim_4_CF["V_rotura_2"])
V_reset_sim_4_CF_all = abs(df_resultados_sim_4_CF["V_rotura_4"])

V_reset_exp_der = abs(df_resultados_exp["V_reset_derivada_V"])
V_reset_max_int = abs(df_resultados_exp["V_reset_max_intensidad_V"])

# ECDF objects
ecdf_sim_2_CF_1 = ECDF(V_reset_sim_2_CF_1)
ecdf_sim_2_CF_2 = ECDF(V_reset_sim_2_CF_2)

ecdf_sim_4_CF_1 = ECDF(V_reset_sim_4_CF_1)
ecdf_sim_4_CF_2 = ECDF(V_reset_sim_4_CF_2)
ecdf_sim_4_CF_all = ECDF(V_reset_sim_4_CF_all)

ecdf_exp_der = ECDF(V_reset_exp_der)
ecdf_exp_max_int = ECDF(V_reset_max_int)

fig, ax = plt.subplots(figsize=(12, 9))
config_ax_IV(ax)

# Common range
min_val = 0
max_val = 1.4
x = np.linspace(min_val, max_val, 1000)

# Evaluate 2 CF ECDFs
y_sim_2_CF_1 = ecdf_sim_2_CF_1(x)
y_sim_2_CF_2 = ecdf_sim_2_CF_2(x)

print(y_sim_2_CF_1)

# Evaluate 4 CF ECDFs
y_sim_4_CF_1 = ecdf_sim_4_CF_1(x)
y_sim_4_CF_2 = ecdf_sim_4_CF_2(x)
y_sim_4_CF_all = ecdf_sim_4_CF_all(x)

# Evaluate experimental ECDFs
y_der = ecdf_exp_der(x)
y_max_int = ecdf_exp_max_int(x)

# Plot ECDFs
ax.step(x, y_sim_4_CF_1, where="post", label="Sim. - 1 CF (4 CF)", linewidth=2.5, linestyle='dashed')
ax.step(x, y_sim_4_CF_2, where="post", label="Sim. - 2 CF (4 CF)", linewidth=2.5, linestyle='dashed')
ax.step(
    x,
    y_sim_4_CF_all,
    where="post",
    label="Simulation - All CF (4 CF)",
    linewidth=2,
    linestyle="dashed",
)

ax.step(x, y_sim_2_CF_1, where="post", label="Sim. - 1 CF (2 CF)", linewidth=2.5)
ax.step(x, y_sim_2_CF_2, where="post", label="Sim. - 2 CF (2 CF)", linewidth=2.5)

ax.scatter(
    x, y_der, label="Exp. - Derivative", linewidth=1, s=50, marker="o"
)
ax.scatter(x, y_max_int, label="Exp. - Current max", linewidth=1, s=50, marker='s')

# Labels in English
ax.set_xlabel(r"|Reset voltage| (\si{\volt})", fontsize=32)
ax.set_ylabel("CDF", fontsize=32)
# ax.set_xlim(0, 1.4)
ax.legend(loc="best", fontsize=26)
plt.savefig(str(Path.cwd()) + "/CDF_reset_ambos.pdf", dpi=300, bbox_inches="tight")
plt.close(fig)

[0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0